# 01 · Análisis exploratorio del dataset ampliado

EP2 · Machine Learning · Waymo Open Dataset v2.0.1 (`lidar_box`)

**Objetivo de este notebook:** entender los datos con los que vamos a comparar modelos: tamaño, calidad, balance de clases y relaciones entre variables. Lo que encontremos aquí justifica las decisiones de preparación y de validación de los notebooks siguientes.

In [ ]:
import sys
sys.path.append("..")  # permite importar el paquete src/ desde notebooks/

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_preprocessing import (
    RANDOM_STATE,
    agregar_distancia_vehiculo,
    agregar_nombre_tipo,
    cargar_segmentos,
)

sns.set_theme(style="whitegrid")
PLOTS = Path("../results/plots")
PLOTS.mkdir(parents=True, exist_ok=True)
ORDEN_CLASES = ["Vehículo", "Señal", "Peatón", "Ciclista"]

## 1. Carga del dataset

En la EP1 trabajamos con un solo segmento. Ahora usamos **30 segmentos del split `training`**, elegidos al azar con semilla 42, para tener todas las clases y suficientes datos para comparar modelos.

In [ ]:
df = cargar_segmentos("../data/raw/lidar_box")
df = agregar_nombre_tipo(df)
df = agregar_distancia_vehiculo(df)

print(f"Filas: {len(df):,} | Columnas: {df.shape[1]} | Segmentos: {df['key_segment_context_name'].nunique()}")
print(f"Objetos únicos (laser_object_id): {df['key_laser_object_id'].nunique():,}")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
df.head()

## 2. Distribución de clases (variable objetivo de la clasificación)

In [ ]:
conteo = df["type_nombre"].value_counts().reindex(ORDEN_CLASES).fillna(0).astype(int)
porcentaje = (conteo / len(df) * 100).round(2)
print(pd.DataFrame({"detecciones": conteo, "porcentaje": porcentaje}))

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=conteo.index, y=conteo.values, ax=ax, color="#3D5AFE")
for i, v in enumerate(conteo.values):
    ax.text(i, v, f"{v:,}\n({porcentaje.iloc[i]}%)", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Detecciones")
ax.set_xlabel("")
ax.set_title("Distribución de clases")
ax.set_ylim(0, max(conteo.max() * 1.18, 1))
fig.savefig(PLOTS / "01_distribucion_clases.png", dpi=150, bbox_inches="tight")
plt.show()

**Hallazgo:** _(escríbelo con tus palabras después de ver el gráfico: qué clases dominan, qué tan rara es la clase Ciclista y qué implica para las métricas)._

## 3. Calidad de los datos

In [ ]:
nulos = df.isnull().sum()
nulos = nulos[nulos > 0]
print("Nulos por columna:\n", nulos, "\n")
print("% nulos:\n", (nulos / len(df) * 100).round(1))
print("\nDuplicados exactos:", df.duplicated().sum())
print("Duplicados objeto+frame:", df.duplicated(subset=["key_laser_object_id", "key_frame_timestamp_micros"]).sum())

**Hallazgo:** _(compara con lo que encontramos en la EP1: ¿se mantienen los nulos de `difficulty_level`? ¿hay duplicados?)._

## 4. Variable objetivo de la regresión: distancia entre el vehículo y cada objeto

Las cajas de `lidar_box` están en el sistema de referencia del vehículo, así que la distancia es la norma del centro de la caja: `sqrt(x² + y² + z²)`. En el modelo **no** usaremos `box_center_x/y/z` como variables de entrada, porque con ellas la distancia se calcula directamente.

In [ ]:
print(df["distancia_vehiculo"].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["distancia_vehiculo"], bins=60, ax=axes[0], color="#3D5AFE")
axes[0].set_title("Distancia al vehículo (m)")
axes[0].set_xlabel("metros")
sns.boxplot(data=df, x="type_nombre", y="distancia_vehiculo", order=ORDEN_CLASES, ax=axes[1])
axes[1].set_title("Distancia por clase")
axes[1].set_xlabel("")
axes[1].set_ylabel("metros")
fig.savefig(PLOTS / "01_distancia.png", dpi=150, bbox_inches="tight")
plt.show()

**Hallazgo:** _(rango de distancias, forma de la distribución y si cambia según la clase)._

## 5. ¿Se puede estimar la distancia a partir de lo que mide el LiDAR?

A mayor distancia, menos puntos del LiDAR caen dentro de la caja de un objeto. Esa es la relación física en la que se apoya la regresión.

In [ ]:
muestra = df.sample(min(20_000, len(df)), random_state=RANDOM_STATE).copy()
muestra["puntos"] = muestra["num_lidar_points_in_box"].clip(lower=1)  # para poder usar escala log

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=muestra, x="distancia_vehiculo", y="puntos", hue="type_nombre",
                hue_order=[c for c in ORDEN_CLASES if c in muestra["type_nombre"].unique()],
                s=8, alpha=0.4, ax=ax)
ax.set_yscale("log")
corr = np.corrcoef(np.log1p(df["num_lidar_points_in_box"]), df["distancia_vehiculo"])[0, 1]
ax.set_title(f"Puntos LiDAR vs distancia (correlación log-puntos/distancia = {corr:.2f})")
ax.set_ylabel("puntos LiDAR en la caja (escala log)")
ax.set_xlabel("distancia al vehículo (m)")
fig.savefig(PLOTS / "01_puntos_vs_distancia.png", dpi=150, bbox_inches="tight")
plt.show()

**Hallazgo:** _(¿la relación es fuerte o dispersa? ¿qué clase se aparta del patrón?)._

## 6. ¿Qué distingue a cada clase? Tamaño y rapidez

Esto anticipa qué variables le servirán a los clasificadores.

In [ ]:
rapidez = np.sqrt(df["speed_x"] ** 2 + df["speed_y"] ** 2 + df["speed_z"] ** 2)
datos = df.assign(rapidez=rapidez)
variables = {"box_size_x": "Largo (m)", "box_size_y": "Ancho (m)",
             "box_size_z": "Alto (m)", "rapidez": "Rapidez (m/s)"}

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (col, titulo) in zip(axes.ravel(), variables.items()):
    sns.boxplot(data=datos, x="type_nombre", y=col, order=ORDEN_CLASES, showfliers=False, ax=ax)
    ax.set_title(titulo)
    ax.set_xlabel("")
    ax.set_ylabel("")
fig.tight_layout()
fig.savefig(PLOTS / "01_tamano_rapidez_por_clase.png", dpi=150, bbox_inches="tight")
plt.show()

**Hallazgo:** _(qué clases se separan fácilmente por tamaño y cuáles se parecen)._

## 7. Estructura de los datos: segmentos y objetos repetidos

Un mismo objeto aparece en muchos frames consecutivos. Si mezclamos esas filas al azar entre entrenamiento y prueba, el modelo vería en entrenamiento al mismo objeto que luego evalúa, y las métricas saldrían infladas. Por eso la validación cruzada se hará **agrupando por segmento**.

In [ ]:
por_segmento = (df.groupby("key_segment_context_name")["type_nombre"].value_counts()
                .unstack(fill_value=0).reindex(columns=ORDEN_CLASES, fill_value=0))
print(por_segmento.describe().round(1))
print("\nSegmentos sin ciclistas:", (por_segmento["Ciclista"] == 0).sum(), "de", len(por_segmento))
print("Segmentos sin peatones:", (por_segmento["Peatón"] == 0).sum(), "de", len(por_segmento))

filas_por_objeto = df.groupby("key_laser_object_id").size()
print(f"\nFrames por objeto: mediana={filas_por_objeto.median():.0f}, máximo={filas_por_objeto.max()}")

objetos_por_clase = (df.groupby("type_nombre")["key_laser_object_id"].nunique()
                     .reindex(ORDEN_CLASES).fillna(0).astype(int))
print("\nObjetos únicos por clase:")
print(objetos_por_clase)

**Hallazgo:** _(cuántos objetos únicos reales hay de la clase Ciclista, y qué implica para evaluar un modelo sobre esa clase)._

## 8. Conclusiones del EDA y decisiones para el modelado

_(complétalo al final, con tus palabras, en 4 o 5 líneas: calidad de los datos, desbalance, variables útiles, la decisión de validar agrupando por segmento y la métrica que usarás para no depender solo del accuracy)._